# 05 · Choosing the Regularization Strength

Notebook 04 left one thing to the eye: *how much* to smooth. Picking $\lambda$
by eye is not reproducible. This notebook replaces the guess with three standard,
quantitative criteria — the **L-curve**, **ABIC**, and **cross-validation** —
all built into `geodef`.

## Learning objectives
- Understand the misfit–roughness trade-off as $\lambda$ varies.
- Locate a good $\lambda$ with the **L-curve** corner.
- Minimize the **Akaike Bayesian Information Criterion (ABIC)**.
- Choose $\lambda$ by **k-fold cross-validation**.
- Compare what the three criteria pick.

## The trade-off

As $\lambda$ grows, the data misfit $\lVert G\mathbf m - \mathbf d\rVert$ rises
(we fit the data less well) while the model roughness $\lVert L\mathbf m\rVert$
falls (the slip gets smoother). We want the sweet spot that fits the data to
about its noise level without inventing structure the data do not require.

- **L-curve.** Plot model norm vs. misfit on log–log axes as $\lambda$ sweeps.
  The curve is L-shaped; its **corner** is the best compromise — past it, misfit
  grows fast for little smoothing gain.
- **ABIC.** Treat regularization as a Bayesian *prior* and $\lambda$ as a
  hyperparameter. ABIC is (minus twice) the marginal likelihood; the
  $\lambda$ that **minimizes ABIC** is the most probable given the data.
- **Cross-validation.** Hold out some data, fit the rest, and predict the held-out
  points. The $\lambda$ that best predicts *unseen* data (minimum CV error)
  generalizes best.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## The L-curve

`geodef.lcurve()` sweeps $\lambda$ and returns the misfit/model-norm arrays plus
the located corner. Its `.plot()` marks the corner.

In [2]:
lc = geodef.lcurve(fault, gnss, smoothing="laplacian", components="dip",
                   smoothing_range=(1e-2, 1e3), n=40)
lc.plot()
print(f"L-curve corner: lambda = {lc.optimal:.3g}")

L-curve corner: lambda = 3.67


## ABIC

`geodef.abic_curve()` evaluates ABIC across the same range. The minimum is the
preferred strength.

In [3]:
ab = geodef.abic_curve(fault, gnss, smoothing="laplacian", components="dip",
                       smoothing_range=(1e-2, 1e3), n=40)
ab.plot()
print(f"ABIC minimum:   lambda = {ab.optimal:.3g}")

ABIC minimum:   lambda = 1.13


## Cross-validation

Pass `smoothing_strength='cv'` and `invert()` runs k-fold cross-validation
internally, choosing the $\lambda$ that best predicts held-out stations.

In [4]:
cv = geodef.invert(fault, gnss, components="dip",
                   smoothing="laplacian", smoothing_strength="cv", cv_folds=5)
print(f"CV-selected:    lambda = {cv.smoothing_strength:.3g}")

CV-selected:    lambda = 0.829


## Do they agree?

The three criteria are derived from different principles, so they need not land
on the same number — but they usually agree to within a factor of a few. Here are
all three choices and the slip each produces.

The top row is the slip each criterion produces (shared symmetric scale); the bottom row is each one's difference from the truth (recovered $-$ true). The residual maps are nearly identical, confirming the criteria land on very similar models.

In [5]:
choices = {
    "L-curve": lc.optimal,
    "ABIC": ab.optimal,
    "cross-val": cv.smoothing_strength,
}
truth = slip_true[N:]
vmax = truth.max()
ncol = len(choices) + 1
fig, axes = plt.subplots(2, ncol, figsize=(3 * ncol, 6),
                         constrained_layout=True)

# Row 0: true + each method's slip, shared symmetric RdBu_r scale.
geodef.plot.slip(fault, truth, ax=axes[0, 0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
recovered = {}
for ax, (name, lam) in zip(axes[0, 1:], choices.items()):
    r = geodef.invert(fault, gnss, components="dip",
                      smoothing="laplacian", smoothing_strength=lam)
    recovered[name] = r.slip_vector
    geodef.plot.slip(fault, r.slip_vector, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"{name}\nlambda={lam:.2g}")

# Row 1: method - true, so the (small) differences between methods show.
diffs = {name: m - truth for name, m in recovered.items()}
dmax = max(np.abs(d).max() for d in diffs.values())
axes[1, 0].axis("off")
for ax, (name, d) in zip(axes[1, 1:], diffs.items()):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"{name} - true")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Exercises
1. **Compare.** Print the three $\lambda$ values side by side. How far apart are
   they? Does the recovered slip look meaningfully different?
2. **Noise sensitivity.** Increase `sigma` to `0.03` and re-run all three. Which
   criterion moves the most?
3. **Damping.** Repeat the L-curve with `smoothing='damping'`. Is the corner as
   sharp?

## Checkpoint questions
1. Why is the corner of the L-curve a sensible compromise?
2. What is ABIC trying to maximize, in Bayesian terms?
3. What does cross-validation optimize that the L-curve and ABIC do not directly?

## Summary
- The misfit–roughness trade-off is controlled by $\lambda$.
- L-curve, ABIC, and CV each pick $\lambda$ by a different, reproducible rule.
- `geodef.lcurve`, `geodef.abic_curve`, and `smoothing_strength='abic'|'cv'`
  automate all three.

**Next:** notebook 06 combines complementary datasets — GNSS and InSAR — in a
single joint inversion.